In [1]:
import sys
import time
import pandas as pd
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
print(f"Running on device: {'GPU' if device == 0 else 'CPU'}")

models_to_test = {
    "BERT (Standard)": "dslim/bert-base-NER",
    "BERT (Uncased)": "dslim/bert-base-NER-uncased"
}

pipelines = {}

for name, model_id in models_to_test.items():
    print(f"Loading {name}...")
    try:
        pipelines[name] = pipeline(
            "token-classification", 
            model=model_id, 
            aggregation_strategy="simple", 
            device=device
        )
    except Exception as e:
        print(f"Failed to load {name}: {e}")

/opt/conda/envs/sentinel-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on device: CPU
Loading BERT (Standard)...


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Loading BERT (Uncased)...


Some weights of the model checkpoint at dslim/bert-base-NER-uncased were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [2]:
# 3. Test Data
test_sentences = [
    # A. Standard Capitalization (Baseline)
    "Elon Musk announced that SpaceX will launch Starship from Texas.",
    
    # B. Lowercase Trap (The Killer Test)
    # Standard BERT models often fail here.
    "apple is planning to open a new store in new york.",
    "amazon is acquiring a startup in san francisco.",
    
    # C. Tricky/Ambiguous
    "The WHO met in Geneva.", # Acronym
    "Jordan played really well today." # Person or Country?
]

print(f"Testing {len(test_sentences)} sentences...")

Testing 5 sentences...


In [3]:
# 4. Run Benchmark
results = []

for text in test_sentences:
    row = {"Text": text}
    
    for model_name, pipe in pipelines.items():
        entities = pipe(text)
        
        # Format entities: "Word (Label)"
        formatted_ents = [f"{e['word']} ({e['entity_group']})" for e in entities]
        row[f"{model_name} Entities"] = formatted_ents
    
    results.append(row)

# 5. Display Results
df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
display(df)

# 6. Automatic Verdict
lowercase_row = df[df["Text"].str.startswith("apple")]

if not lowercase_row.empty:
    std_ents = lowercase_row.iloc[0]["BERT (Standard) Entities"]
    uncased_ents = lowercase_row.iloc[0]["BERT (Uncased) Entities"]
    
    # Check if 'apple' was detected as ORG
    std_success = any("apple" in e.lower() and "ORG" in e for e in std_ents)
    uncased_success = any("apple" in e.lower() and "ORG" in e for e in uncased_ents)


,Text,BERT (Standard) Entities,BERT (Uncased) Entities
0,Elon Musk announced that SpaceX will launch Starship from Texas.,"[Elon Musk (ORG), SpaceX (ORG), Starship (MISC), Texas (LOC)]","[elon musk (ORG), spacex (ORG), texas (LOC)]"
1,apple is planning to open a new store in new york.,[],"[apple (ORG), new york (LOC)]"
2,amazon is acquiring a startup in san francisco.,"[am (ORG), f (LOC)]","[amazon (ORG), san francisco (LOC)]"
3,The WHO met in Geneva.,"[WHO (ORG), Geneva (LOC)]","[who (ORG), geneva (LOC)]"
4,Jordan played really well today.,[Jordan (ORG)],[jordan (LOC)]
